# gen-gec-errant — Batch Run All Models: KUPA-KEYS (Colab)

Run the full **generate → GEC → ERRANT → analysis** pipeline for **all models**
in the registry on the **KUPA-KEYS** dataset.

**Use case:** Full paper reproduction. Runs sequentially through each model,
saving results to Google Drive after each. Designed for long Colab Pro sessions.

**Data source:** `phd-experimental-data/cefr-classification/data/splits/norm-KUPA-KEYS.csv`  
**Models:** `_p/artificial-learners/models/` (fine-tuned) + HuggingFace Hub (native)  
**Output:** `_p/artificial-learners/generation-gec-errant/results/batch-kupa-keys-<timestamp>/`  

**Runtime:** ~1–3 hours total on T4 GPU (depends on `MAX_SENTENCES`).

*Last updated: 2026-03-16*

## 0. Mount Google Drive & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%time
!pip install torch transformers errant spacy numpy scipy matplotlib pandas pyyaml accelerate -q
!python -m spacy download en_core_web_sm -q

In [ ]:
%%time
# Install gen_gec_errant from GitHub (public repo)
!pip install git+https://github.com/berstearns/public-automatic-generation-correction-errortagging-v2.git -q

## 1. Configuration

Configure which models to run and dataset limits.

Set `MODELS_TO_RUN` to `None` to run all models, or pass a list of model keys to run a subset.

In [ ]:
from pathlib import Path
from datetime import datetime
import torch

from gen_gec_errant.registry import get_models, get_datasets, build_pipeline_config, PathConfig
from gen_gec_errant.colab import resolve_model_path, cleanup_local_model

# ══════════════════════════════════════════════════════════════════════
#  USER CONFIG — edit these
# ══════════════════════════════════════════════════════════════════════
GDRIVE = Path('/content/drive/MyDrive')

DATA_ROOT    = GDRIVE / 'phd-experimental-data/cefr-classification/data/splits'
MODELS_ROOT  = GDRIVE / '_p/artificial-learners/models'
OUTPUT_ROOT  = GDRIVE / '_p/artificial-learners/generation-gec-errant/results'

MODELS_TO_RUN  = None                   # None = all models; or ['ft-gpt2-small', 'ft-pythia-70m']
DATASETS_TO_RUN = ['norm-KUPA-KEYS']    # which datasets to run
MAX_SENTENCES  = None                   # None = all data; 50 for quick test
INCLUDE_LEARNER_BASELINE = True
# ══════════════════════════════════════════════════════════════════════

# Derived
TIMESTAMP = datetime.now().strftime('%Y%m%d-%H%M%S')
ds_tag = '-'.join(d.replace('norm-', '').lower() for d in DATASETS_TO_RUN)
BATCH_RUN_DIR = OUTPUT_ROOT / f'batch-{ds_tag}-{TIMESTAMP}'

paths = PathConfig(data_root=DATA_ROOT, models_root=MODELS_ROOT, output_root=OUTPUT_ROOT)
models = get_models(MODELS_TO_RUN)
datasets = get_datasets(DATASETS_TO_RUN)

# ── Device ──
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print(f'\nOutput dir:  {BATCH_RUN_DIR}')
print(f'Data root:   {DATA_ROOT}')
print(f'Models root: {MODELS_ROOT}')
print(f'Models: {len(models)}  |  Datasets: {len(datasets)}  |  Max sentences: {MAX_SENTENCES or "all"}')

## 2. Run All Models

Iterates through each model, copies weights to local SSD, runs the pipeline for each dataset,
saves results to GDrive, and cleans up local model cache before moving to the next model.

In [ ]:
import logging
import time
import json
import gc

logging.basicConfig(level=logging.INFO, format='%(name)s | %(message)s')

from gen_gec_errant.pipeline import run_pipeline

In [ ]:
%%time

BATCH_RUN_DIR.mkdir(parents=True, exist_ok=True)

# Track progress across all models
progress_log = []
t_batch_start = time.time()

for model_idx, model in enumerate(models, 1):
    print('\n' + '#' * 70)
    print(f'# MODEL {model_idx}/{len(models)}: {model.name} ({model.params})')
    print(f'# {model.description}')
    print('#' * 70)

    # Resolve model path (copy to local SSD if fine-tuned)
    model_path = resolve_model_path(model, paths)
    if model_path is None:
        progress_log.append({'model': model.name, 'status': 'skipped', 'reason': 'model not found'})
        continue

    model_results = {}

    for ds in datasets:
        ds_path = paths.dataset_path(ds)
        print(f'\n  --- {ds.name} ---')

        if not ds_path.exists():
            print(f'  SKIPPED: file not found')
            continue

        output_dir = BATCH_RUN_DIR / model.name / ds.name
        raw_results = output_dir / 'raw_results.json'

        if raw_results.exists():
            print(f'  Already completed. Skipping.')
            continue

        config = build_pipeline_config(
            model, ds, paths,
            model_path=model_path,
            max_sentences=MAX_SENTENCES,
            include_learner_baseline=INCLUDE_LEARNER_BASELINE,
            output_dir=str(output_dir),
        )

        t0 = time.time()
        summaries, comparison = run_pipeline(config)
        elapsed = time.time() - t0

        model_results[ds.name] = {'summaries': summaries, 'elapsed': elapsed}
        print(f'  Completed in {elapsed / 60:.1f} min')

        # Free GPU memory between datasets
        if device == 'cuda':
            torch.cuda.empty_cache()

    # Build cross-dataset summary for this model
    cross_summary = {}
    for ds in datasets:
        output_dir = BATCH_RUN_DIR / model.name / ds.name
        entry = {'dataset': ds.name}

        model_summary_path = output_dir / f'{model.name}_summary.json'
        baseline_path = output_dir / 'learner_baseline_summary.json'

        if model_summary_path.exists():
            with open(model_summary_path) as f:
                entry['model'] = json.load(f)
        if baseline_path.exists():
            with open(baseline_path) as f:
                entry['learner_baseline'] = json.load(f)

        cross_summary[ds.name] = entry

    summary_path = BATCH_RUN_DIR / model.name / 'cross_dataset_summary.json'
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    with open(summary_path, 'w') as f:
        json.dump(cross_summary, f, indent=2, default=str)

    progress_log.append({'model': model.name, 'status': 'completed', 'datasets': len(model_results)})

    # Cleanup local model cache to free disk
    cleanup_local_model(model)
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

total_elapsed = time.time() - t_batch_start
print(f'\n{"=" * 70}')
print(f'BATCH RUN COMPLETE in {total_elapsed / 60:.1f} min ({total_elapsed / 3600:.1f} hours)')
print(f'Results: {BATCH_RUN_DIR}')
print(f'{"=" * 70}')

## 3. Aggregate Cross-Model Summary

In [ ]:
import pandas as pd

ds_display = {ds.name: ds.name.replace('norm-', '') for ds in datasets}

# Build a table: rows=models, cols=dataset metrics
rows = []

for model in models:
    summary_path = BATCH_RUN_DIR / model.name / 'cross_dataset_summary.json'
    if not summary_path.exists():
        continue

    with open(summary_path) as f:
        cross = json.load(f)

    row = {
        'model': model.name,
        'params': model.params,
        'family': model.model_family,
        'learner_tuned': model.is_learner_tuned,
    }

    for ds_name, ds_data in cross.items():
        ds_short = ds_display.get(ds_name, ds_name)
        m = ds_data.get('model', {})
        row[f'{ds_short}_ppl'] = m.get('ppl_mean')
        row[f'{ds_short}_err_rate'] = m.get('error_rate')
        row[f'{ds_short}_err_sent'] = m.get('avg_errors_per_sentence')

    rows.append(row)

if rows:
    df = pd.DataFrame(rows)
    print('Cross-Model Summary (Perplexity):')
    ppl_cols = ['model', 'params', 'family', 'learner_tuned'] + [c for c in df.columns if '_ppl' in c]
    print(df[ppl_cols].to_string(index=False))

    print('\nCross-Model Summary (Errors per Sentence):')
    err_cols = ['model', 'params'] + [c for c in df.columns if '_err_sent' in c]
    print(df[err_cols].to_string(index=False))

    # Save
    df.to_csv(BATCH_RUN_DIR / 'cross_model_summary.csv', index=False)
    print(f'\nSaved: {BATCH_RUN_DIR / "cross_model_summary.csv"}')
else:
    print('No completed results found.')

## 4. Cross-Model Visualizations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if not rows:
    print('No results to plot.')
else:
    n_ds = len(ds_display)

    # ── Perplexity by Model ────────────────────────────────────────────
    fig, axes = plt.subplots(1, n_ds, figsize=(6 * n_ds, 6), squeeze=False)
    axes = axes[0]

    for ax, (ds_name, ds_short) in zip(axes, ds_display.items()):
        col = f'{ds_short}_ppl'
        if col not in df.columns:
            continue
        valid = df[df[col].notna()].sort_values(col)

        colors = ['#DD8452' if lt else '#4C72B0' for lt in valid['learner_tuned']]
        ax.barh(valid['model'], valid[col], color=colors)
        ax.set_xlabel('Mean Perplexity')
        ax.set_title(ds_short)
        ax.invert_yaxis()

    fig.suptitle('Perplexity by Model and Dataset', fontsize=14)
    plt.tight_layout()
    fig.savefig(str(BATCH_RUN_DIR / 'perplexity_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── Error Rate by Model ───────────────────────────────────────────
    fig, axes = plt.subplots(1, n_ds, figsize=(6 * n_ds, 6), squeeze=False)
    axes = axes[0]

    for ax, (ds_name, ds_short) in zip(axes, ds_display.items()):
        col = f'{ds_short}_err_sent'
        if col not in df.columns:
            continue
        valid = df[df[col].notna()].sort_values(col)

        colors = ['#DD8452' if lt else '#4C72B0' for lt in valid['learner_tuned']]
        ax.barh(valid['model'], valid[col], color=colors)
        ax.set_xlabel('Avg Errors per Sentence')
        ax.set_title(ds_short)
        ax.invert_yaxis()

    fig.suptitle('Error Rate by Model and Dataset\n(orange=learner-tuned, blue=native)', fontsize=14)
    plt.tight_layout()
    fig.savefig(str(BATCH_RUN_DIR / 'error_rate_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Plots saved to {BATCH_RUN_DIR}')

## 5. Progress & Diagnostics

In [ ]:
import yaml

# Show progress log
print('Progress Log:')
print(f'{"Model":<25} {"Status":<12} {"Details"}')
print(f'{"-"*25} {"-"*12} {"-"*20}')
for entry in progress_log:
    details = entry.get('reason', f'{entry.get("datasets", 0)} datasets')
    print(f'{entry["model"]:<25} {entry["status"]:<12} {details}')

# Save progress log
log_path = BATCH_RUN_DIR / 'progress_log.json'
with open(log_path, 'w') as f:
    json.dump(progress_log, f, indent=2)
print(f'\nProgress log saved: {log_path}')

# Save batch config
from gen_gec_errant.registry import PIPELINE_DEFAULTS
batch_config = {
    'models_run': [m.name for m in models],
    'max_sentences': MAX_SENTENCES,
    'gec_model': PIPELINE_DEFAULTS['gec']['model_id'],
    'seed': PIPELINE_DEFAULTS['seed'],
    'timestamp': TIMESTAMP,
    'device': device,
    'gpu': torch.cuda.get_device_name() if device == 'cuda' else None,
}
with open(BATCH_RUN_DIR / 'batch_config.yaml', 'w') as f:
    yaml.dump(batch_config, f, default_flow_style=False)

print(f'\n=== All artifacts saved to {BATCH_RUN_DIR} ===')